# trainer-class-skeleton — ex2: Trainer with on_epoch_end callbacks list

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `trainer-class-skeleton`. Running the final beacon cell reports progress against the `Trainer: Trainer class skeleton` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Trainer: Trainer class skeleton` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`trainer-class-skeleton`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "trainer-class-skeleton"
DD_SUBTOPIC = "Trainer: Trainer class skeleton"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Trainer callbacks — quick refresher

Once `fit / validate / _step` work, the next extension point is POST-EPOCH HOOKS. Lightning calls them callbacks; ARENA calls them "on_epoch_end" functions. The pattern: a list of callables that the Trainer invokes after each epoch, receiving the trainer itself so they can read `self.step`, `self.history`, `self.model`, etc.

```python
class Trainer:
    def __init__(self, ...):
        ...
        self.callbacks = []          # list of fn(trainer) -> None

    def fit(self, n_epochs):
        for epoch in range(n_epochs):
            self._train_epoch()
            self.validate()
            for cb in self.callbacks:
                cb(self)             # hook point — runs AFTER validate
```

This is the right hook point because validate has already appended this epoch's val loss to `history`, so a callback can read it (early stopping, LR scheduling, checkpointing all work here).

### Exercise 2 — Trainer with on_epoch_end callbacks list

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the callback-list extension to the Trainer skeleton: a `self.callbacks` list of `fn(trainer)` callables that fire after each `validate()` call inside `fit`.
> Keywords: callbacks, on-epoch-end, hooks, trainer-extension
> ```

**KCs targeted:** `trainer-fit-loop-walks-epochs`, `trainer-callbacks-fire-after-validate`

Implement `Ex2TrainerWithCallbacks`. Same shape as the minimal Trainer from ex1, plus a callbacks list.

1. `__init__(self, model, optimizer, train_loader, val_loader, loss_fn)`:
   - Store the five args; `self.step = 0`; `self.history = {'train_loss': [], 'val_loss': []}`.
   - **NEW:** `self.callbacks = []` — list of `fn(trainer) -> None` callables.

2. `_step(self, x, y)`: forward + loss (`self.loss_fn(self.model(x), y)`).

3. `fit(self, n_epochs)`: per epoch:
   - `self.model.train()`.
   - iterate train_loader: `loss = self._step(x, y); loss.backward(); optimizer.step(); optimizer.zero_grad(); self.step += 1; history['train_loss'].append(loss.item())`.
   - `self.validate()`.
   - **NEW:** `for cb in self.callbacks: cb(self)` — runs AFTER validate so callbacks see this epoch's `history['val_loss']`.

4. `validate(self)`: eval mode + inference_mode; weighted-by-batch-size val loss appended to `history['val_loss']`.

The test registers TWO callbacks: an EpochCounter (just counts calls) and an EarlyStopRecorder (snapshots `history['val_loss'][-1]` after each epoch). The test verifies both fire exactly `n_epochs` times AND that the val-loss snapshot is the value freshly-appended by THIS epoch's validate (not stale).

In [ ]:
class Ex2TrainerWithCallbacks:
    """Trainer + on_epoch_end callback list."""

    def __init__(self, model, optimizer, train_loader, val_loader, loss_fn):
        raise NotImplementedError()

    def _step(self, x, y):
        raise NotImplementedError()

    def fit(self, n_epochs: int):
        raise NotImplementedError()

    def validate(self):
        raise NotImplementedError()


def _test_ex2():
    from torch.utils.data import TensorDataset, DataLoader

    t.manual_seed(0)
    N = 64
    x_train = t.randn(N, 1)
    y_train = 2.0 * x_train + 1.0 + 0.05 * t.randn(N, 1)
    x_val = t.randn(16, 1)
    y_val = 2.0 * x_val + 1.0

    train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=8, shuffle=True)
    val_loader = DataLoader(TensorDataset(x_val, y_val), batch_size=8, shuffle=False)

    model = t.nn.Linear(1, 1)
    opt = t.optim.SGD(model.parameters(), lr=0.1)
    loss_fn = t.nn.MSELoss()

    trainer = Ex2TrainerWithCallbacks(model, opt, train_loader, val_loader, loss_fn)

    # Required attributes including the new callbacks list.
    assert hasattr(trainer, 'callbacks'), 'must have self.callbacks list'
    assert trainer.callbacks == [], f'callbacks must start empty; got {trainer.callbacks}'
    assert trainer.step == 0
    assert trainer.history == {'train_loss': [], 'val_loss': []}

    # === Register two callbacks ===
    epoch_count = {'n': 0}
    def epoch_counter(trainer_self):
        epoch_count['n'] += 1

    val_snapshots = []
    def early_stop_recorder(trainer_self):
        # Read the fresh val-loss appended by THIS epoch's validate.
        val_snapshots.append(trainer_self.history['val_loss'][-1])

    trainer.callbacks.append(epoch_counter)
    trainer.callbacks.append(early_stop_recorder)

    # === Fit 4 epochs ===
    trainer.fit(n_epochs=4)

    # Each callback fired exactly n_epochs times.
    assert epoch_count['n'] == 4, f'epoch_counter should fire 4×; got {epoch_count["n"]}'
    assert len(val_snapshots) == 4, f'early_stop_recorder should fire 4×; got {len(val_snapshots)}'

    # Snapshots must MATCH history['val_loss'] (callbacks ran AFTER validate).
    assert val_snapshots == trainer.history['val_loss'], (
        f'callback snapshots must equal history[val_loss]; '
        f'snap={val_snapshots} hist={trainer.history["val_loss"]}'
    )

    # Validate that step counter and training loss history are right.
    expected_steps = 4 * len(train_loader)
    assert trainer.step == expected_steps
    assert len(trainer.history['train_loss']) == expected_steps
    assert len(trainer.history['val_loss']) == 4

    # === Loss decreased ===
    assert trainer.history['val_loss'][-1] < trainer.history['val_loss'][0], (
        'val loss should decrease across 4 epochs'
    )

    # === No-callback path still works ===
    t.manual_seed(0)
    model2 = t.nn.Linear(1, 1)
    opt2 = t.optim.SGD(model2.parameters(), lr=0.1)
    trainer2 = Ex2TrainerWithCallbacks(model2, opt2, train_loader, val_loader, loss_fn)
    # Don't register any callbacks.
    trainer2.fit(n_epochs=2)
    assert len(trainer2.history['val_loss']) == 2, 'fit must work with empty callbacks list'

    # === Callback fires AFTER validate (sees the fresh val_loss, not stale) ===
    # Re-run with a callback that compares len(history[val_loss]) to the epoch number.
    t.manual_seed(0)
    model3 = t.nn.Linear(1, 1)
    opt3 = t.optim.SGD(model3.parameters(), lr=0.1)
    trainer3 = Ex2TrainerWithCallbacks(model3, opt3, train_loader, val_loader, loss_fn)
    observed_lens = []
    def observe_history_len(self):
        observed_lens.append(len(self.history['val_loss']))
    trainer3.callbacks.append(observe_history_len)
    trainer3.fit(n_epochs=3)
    assert observed_lens == [1, 2, 3], (
        f'callback should fire AFTER validate appends this epoch\'s val_loss; '
        f'expected [1, 2, 3], got {observed_lens}. '
        f'If [0, 1, 2], your callback runs BEFORE validate.'
    )
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
class Ex2TrainerWithCallbacks:
    def __init__(self, model, optimizer, train_loader, val_loader, loss_fn):
        self.model = model
        self.optimizer = optimizer
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.loss_fn = loss_fn
        self.step = 0
        self.history = {'train_loss': [], 'val_loss': []}
        self.callbacks = []

    def _step(self, x, y):
        return self.loss_fn(self.model(x), y)

    def fit(self, n_epochs):
        for _epoch in range(n_epochs):
            self.model.train()
            for x, y in self.train_loader:
                loss = self._step(x, y)
                loss.backward()
                self.optimizer.step()
                self.optimizer.zero_grad()
                self.step += 1
                self.history['train_loss'].append(loss.item())
            self.validate()
            for cb in self.callbacks:
                cb(self)

    def validate(self):
        self.model.eval()
        total = 0.0
        count = 0
        with t.inference_mode():
            for x, y in self.val_loader:
                loss = self.loss_fn(self.model(x), y)
                total += loss.item() * x.shape[0]
                count += x.shape[0]
        self.history['val_loss'].append(total / count)
```

**Why callbacks run AFTER validate.** A callback that wants to trigger early stopping on val loss MUST see this epoch's val loss in `history['val_loss']`. Running before validate would give it stale data (last epoch's loss). The order train → validate → callbacks is the same one PyTorch Lightning uses (`on_validation_end` → `on_epoch_end`).

**Why `cb(self)` and not bound methods.** Passing the trainer itself lets callbacks be plain functions — easiest to write and test. They can read anything on the trainer (history, model, step, optimizer LR groups) without inheritance. Lightning uses the same convention with its `Callback` class taking a `trainer` argument in every hook.

**Common callbacks you'd register in real code.** `ModelCheckpoint(save_best=True)` reads `history['val_loss'][-1]` and saves model weights when it improves. `EarlyStopping(patience=5)` raises a sentinel to break fit early. `LRScheduler.step()` adjusts the optimizer LR based on the new val loss. All three only need the trainer reference and the post-validate moment.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()